## Question 1: Time Series Classification — Feature Extraction

In [3]:
from pathlib import Path
import zipfile

zip_path = Path("/content/AReM.zip")
data_dir = Path("/content/AReM")

with zipfile.ZipFile(zip_path) as archive:
    archive.extractall("/content")

for folder in sorted(data_dir.iterdir()):
    if folder.is_dir():
        print(folder.name, len(list(folder.glob("*.csv"))), "CSV files")

bending1 7 CSV files
bending2 6 CSV files
cycling 15 CSV files
lying 15 CSV files
sitting 15 CSV files
standing 15 CSV files
walking 15 CSV files


In [4]:
train_files = []
test_files = []

for folder in sorted(data_dir.iterdir()):
    if not folder.is_dir():
        continue

    for path in sorted(folder.glob("dataset*.csv")):
        number = int(path.stem.removeprefix("dataset"))
        test_limit = 2 if folder.name in ("bending1", "bending2") else 3

        if number <= test_limit:
            test_files.append(path)
        else:
            train_files.append(path)

print("Training instances:", len(train_files))
print("Test instances:", len(test_files))
print("Total:", len(train_files) + len(test_files))

Training instances: 69
Test instances: 19
Total: 88


In [5]:
import pandas as pd

columns = [
    "time", "avg_rss12", "var_rss12",
    "avg_rss13", "var_rss13", "avg_rss23", "var_rss23"
]

def load_instance(path):
    rows = []

    with open(path) as file:
        for line in file:
            if line.startswith("#") or not line.strip():
                continue
            values = line.strip().replace(",", " ").split()
            if len(values) != 7:
                raise ValueError(f"{path}: expected 7 values, got {len(values)}")
            rows.append([float(value) for value in values])

    return pd.DataFrame(rows, columns=columns)

row_counts = pd.Series({
    str(path.relative_to(data_dir)): len(load_instance(path))
    for path in train_files + test_files
})

print(row_counts.value_counts().sort_index())
print("\nFiles with fewer than 480 rows:")
print(row_counts[row_counts < 480])

479     1
480    87
Name: count, dtype: int64

Files with fewer than 480 rows:
sitting/dataset8.csv    479
dtype: int64


### 1(c)(i): Common Time-Domain Features

Common time-domain features include minimum, maximum, range, mean, median, first and third quartiles, interquartile range, variance, standard deviation, mean or median absolute deviation, root mean square, skewness, kurtosis, mean absolute change, and autocorrelation.

### 1(c)(ii): Feature Extraction

For each instance, I calculated the minimum, maximum, mean, median, standard deviation, first quartile, and third quartile of each of its six sensor time series. This produces 42 features per instance. I kept the activity and filename as identifiers.

In [6]:
sensor_columns = columns[1:]

def extract_features(path):
    data = load_instance(path)
    result = {
        "activity": path.parent.name,
        "file": path.name,
    }

    for i, column in enumerate(sensor_columns, start=1):
        series = data[column]
        result[f"min{i}"] = series.min()
        result[f"max{i}"] = series.max()
        result[f"mean{i}"] = series.mean()
        result[f"median{i}"] = series.median()
        result[f"std{i}"] = series.std()
        result[f"q1_{i}"] = series.quantile(0.25)
        result[f"q3_{i}"] = series.quantile(0.75)

    return result

train_features = pd.DataFrame([extract_features(path) for path in train_files])
test_features = pd.DataFrame([extract_features(path) for path in test_files])
all_features = pd.concat([train_features, test_features], ignore_index=True)

print("Train:", train_features.shape)
print("Test:", test_features.shape)
print("All:", all_features.shape)
display(all_features.head())

Train: (69, 44)
Test: (19, 44)
All: (88, 44)


,activity,file,min1,max1,mean1,median1,std1,q1_1,q3_1,min2,...,std5,q1_5,q3_5,min6,max6,mean6,median6,std6,q1_6,q3_6
0,bending1,dataset3.csv,35.00,47.40,43.954500,44.33,1.558835,43.00,45.00,0.0,...,1.999604,35.3625,36.50,0.0,1.79,0.493292,0.43,0.513506,0.00,0.94
1,bending1,dataset4.csv,33.00,47.75,42.179812,43.50,3.670666,39.15,45.00,0.0,...,3.849448,30.4575,36.33,0.0,2.18,0.613521,0.50,0.524317,0.00,1.00
2,bending1,dataset5.csv,33.00,45.75,41.678063,41.75,2.243490,41.33,42.75,0.0,...,2.411026,28.4575,31.25,0.0,1.79,0.383292,0.43,0.389164,0.00,0.50
3,bending1,dataset6.csv,37.00,48.00,43.454958,43.25,1.386098,42.50,45.00,0.0,...,2.488862,22.2500,24.00,0.0,5.26,0.679646,0.50,0.622534,0.43,0.87
4,bending1,dataset7.csv,36.25,48.00,43.969125,44.50,1.618364,43.31,44.67,0.0,...,3.318301,20.5000,23.75,0.0,2.96,0.555312,0.49,0.487826,0.00,0.83


### 1(c)(iii): Standard Deviations and 90% Bootstrap Confidence Intervals

I estimated the sample standard deviation of each of the 42 feature columns across all 88 instances. I then resampled the 88 instances with replacement 10,000 times and recalculated each standard deviation. The 5th and 95th percentiles of those bootstrap values form the 90% confidence interval.

In [7]:
import numpy as np

feature_columns = all_features.columns.drop(["activity", "file"])
X = all_features[feature_columns].to_numpy()

rng = np.random.default_rng(42)
n_boot = 10_000
boot_std = np.empty((n_boot, len(feature_columns)))

for b in range(n_boot):
    indices = rng.integers(0, len(X), size=len(X))
    boot_std[b] = X[indices].std(axis=0, ddof=1)

results = pd.DataFrame({
    "feature": feature_columns,
    "estimated_std": X.std(axis=0, ddof=1),
    "ci_5%": np.percentile(boot_std, 5, axis=0),
    "ci_95%": np.percentile(boot_std, 95, axis=0),
})

display(results.round(3))

,feature,estimated_std,ci_5%,ci_95%
0,min1,9.570,8.244,10.759
1,max1,4.394,3.327,5.306
2,mean1,5.336,4.706,5.879
3,median1,5.440,4.796,5.994
4,std1,1.772,1.573,1.949
5,q1_1,6.154,5.569,6.639
6,q3_1,5.139,4.327,5.827
7,min2,0.000,0.000,0.000
8,max2,5.063,4.618,5.410
9,mean2,1.574,1.394,1.706


### 1(c)(iv): Three Selected Features
I selected the mean, standard deviation, and maximum. The mean describes the typical sensor reading, standard deviation describes how much it varies during an activity, and maximum captures occasional high readings. Together they represent level, variability, and peaks. I did not select minimum because some minimum features are constant across all instances.

## Question 2: ISLR 3.7.4

**(a)** The cubic model's training RSS is less than or equal to the linear model's training RSS because the cubic model includes the linear model as a special case. Its coefficients for \(X^2\) and \(X^3\) can both be zero.

**(b)** We generally expect the linear model to have lower test RSS because the true relationship is linear. The cubic model's extra terms may fit noise in the training data and generalize less well.

**(c)** The cubic model's training RSS is again less than or equal to the linear model's training RSS. This holds regardless of whether the true relationship is linear because the linear model is a special case of the cubic model.

**(d)** There is not enough information to determine which model has lower test RSS. A cubic model could better approximate the nonlinear relationship, but its extra flexibility can also increase variance. The result depends on the shape of the true relationship and the data.